In [6]:
import numpy as np
import networkx as nx
import math

def generate_dataset(n=10, m=8, lambda_=3, mu_=3):
    np.random.seed(42)
    S = np.random.uniform(0.2, 0.8, (n, m))
    # S[:, 0] = np.random.uniform(0.0, 0.2, n)
    # S[0, 0] = 0.95
    # S[1, 0] = 0.92

    return S, n, m, lambda_, mu_

def peer_review_4_all_simplified(S, lambda_, mu_):
    n, m = S.shape
    assignment = {j: [] for j in range(m)}
    reviewer_caps = {i: mu_ for i in range(n)}
    papers_needed = {j: lambda_ for j in range(m)}

    sims = []
    for i in range(n):
        for j in range(m):
            sims.append((S[i, j], i, j))
    sims.sort(key=lambda x: x[0], reverse=True)

    def check_flow(k_target, active_papers):
        G = nx.DiGraph()
        G.add_node('Source')
        G.add_node('Sink')

        for i in range(n):
            if reviewer_caps[i] > 0:
                G.add_edge('Source', f'R_{i}', capacity=reviewer_caps[i])

        for j in active_papers:
            needed = min(papers_needed[j], k_target)
            G.add_edge(f'P_{j}', 'Sink', capacity=needed)

        for sim, i, j in sims:
            if j in active_papers and reviewer_caps[i] > 0 and i not in assignment[j]:
                G.add_edge(f'R_{i}', f'P_{j}', capacity=1)

            flow_val, flow_dict = nx.maximum_flow(G, 'Source', 'Sink')
            target_flow = sum(min(papers_needed[p], k_target) for p in active_papers)

            if flow_val == target_flow:
                return flow_dict, sim

        return None, -1

    active_papers = list(range(m))
    while active_papers:
        best_flow = None
        best_k = 1

        for k in range(1, lambda_ + 1):
            flow_dict, min_sim = check_flow(k, active_papers)
            if flow_dict is not None:
                best_flow = flow_dict
                best_k = k

        if not best_flow:
            raise ValueError("No feasible flow found. Constraints too tight.")

        temp_scores = {p: 0 for p in active_papers}
        temp_assignment = {p: [] for p in active_papers}
        for i in range(n):
            if f'R_{i}' in best_flow:
                for dest, flow in best_flow[f'R_{i}'].items():
                    if flow > 0 and dest.startswith('P_'):
                        p = int(dest.split('_')[1])
                        temp_scores[p] += S[i, p]
                        temp_assignment[p].append(i)

        worst_paper = min(active_papers, key=lambda p: temp_scores[p])

        for rev in temp_assignment[worst_paper]:
            assignment[worst_paper].append(rev)
            reviewer_caps[rev] -= 1
            papers_needed[worst_paper] -= 1

        if papers_needed[worst_paper] == 0:
            active_papers.remove(worst_paper)

    return assignment

def min_cost_flow_nsw(S, lambda_, mu_, threshold=0.7):

    n, m = S.shape
    B = (S >= threshold).astype(int)

    G = nx.DiGraph()
    G.add_node('Source')
    G.add_node('Sink')


    for i in range(n):
        G.add_edge('Source', f'R_{i}', capacity=mu_, weight=0)

    for j in range(m):
        for k in range(1, lambda_ + 1):

            marginal_utility = math.log(k + 1) - math.log(k)
            cost = int(-1000 * marginal_utility)

            node_name = f'P_{j}_k{k}'
            G.add_edge(f'P_{j}', node_name, capacity=1, weight=0)
            G.add_edge(node_name, 'Sink', capacity=1, weight=cost)

    for i in range(n):
        for j in range(m):
            if B[i, j] == 1:

                G.add_edge(f'R_{i}', f'P_{j}', capacity=1, weight=0)
            else:

                G.add_edge(f'R_{i}', f'P_{j}', capacity=1, weight=5000)


    G.nodes['Source']['demand'] = - (m * lambda_)
    G.nodes['Sink']['demand'] = (m * lambda_)

    flow_cost, flow_dict = nx.network_simplex(G)


    assignment = {j: [] for j in range(m)}
    for i in range(n):
        for dest, flow in flow_dict[f'R_{i}'].items():
            if flow > 0 and dest.startswith('P_'):
                p = int(dest.split('_')[1])
                assignment[p].append(i)

    return assignment

def evaluate_metrics(S, assignment):

    m = S.shape[1]
    valuations = []

    for j in range(m):
        score = sum(S[i, j] for i in assignment[j])
        valuations.append(score)

    fairness = min(valuations)

    nsw = np.prod([max(v, 1e-4) for v in valuations]) ** (1/m)

    return nsw, fairness, sorted(valuations)


S, n, m, lambda_, mu_ = generate_dataset()


assign_pr4a = peer_review_4_all_simplified(S, lambda_, mu_)
assign_nsw = min_cost_flow_nsw(S, lambda_, mu_, threshold=0.4)


nsw_a, fair_a, vals_a = evaluate_metrics(S, assign_pr4a)
nsw_b, fair_b, vals_b = evaluate_metrics(S, assign_nsw)

# Output Results
print("=== PeerReview4All (Algorithm A) ===")
print(f"Fairness (Max-Min) : {fair_a:.4f}")
print(f"Nash Social Welfare: {nsw_a:.4f}")
print(f"Sorted Valuations  : {[round(v, 3) for v in vals_a]}\n")

print("=== Binary NSW via Min-Cost Flow (Algorithm B) ===")
print(f"Fairness (Max-Min) : {fair_b:.4f}")
print(f"Nash Social Welfare: {nsw_b:.4f}")
print(f"Sorted Valuations  : {[round(v, 3) for v in vals_b]}")

=== PeerReview4All (Algorithm A) ===
Fairness (Max-Min) : 1.5623
Nash Social Welfare: 1.9228
Sorted Valuations  : [np.float64(1.562), np.float64(1.785), np.float64(1.909), np.float64(1.971), np.float64(1.985), np.float64(2.043), np.float64(2.064), np.float64(2.127)]

=== Binary NSW via Min-Cost Flow (Algorithm B) ===
Fairness (Max-Min) : 1.5134
Nash Social Welfare: 1.7631
Sorted Valuations  : [np.float64(1.513), np.float64(1.538), np.float64(1.554), np.float64(1.688), np.float64(1.85), np.float64(1.925), np.float64(2.02), np.float64(2.127)]
